# Read H-Reflex App Data Files

This notebook reads and visualizes data from the **H-Reflex Behavior App** (hreflex_txbdc) binary data files.

**V2 file convention (current):**
- **`.hrs1`** — MH Recruitment Curve stage: sweeps across stimulation intensities to map the M/H-wave recruitment curve.
- **`.hrs2`** — Control Mode stage: stimulates at a fixed user-set intensity (can be changed between trials).
- **`.hrs3`** — Down Condition Pellet (DCP) stage: closed-loop H-reflex conditioning with pellet reward.

**V1 file convention (legacy):**
- **`.hrs1`** — EMG Characterization stage: captures baseline EMG grand-mean distribution.
- **`.hrs2`** — MH Recruitment Curve stage (same binary format as V2 `.hrs1`).

All trial files share the MhRecHeader + MhRecTrial binary format.
EMG data blocks (raw differential, filtered, abs-value) are embedded in every file.

The binary format is based on the `FileIO_Helpers` serialization from the `hreflex_txbdc` package.

# Section 1: Binary File Reader Utilities

In [37]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from helpers import (
    # File readers
    read_hrs1, read_hrs2, read_hrs3, find_hrs_files, detect_app_version,
    # Summary printers
    print_hrs1_summary, print_hrs2_summary,
    # HRS2 plots
    plot_amplitude_distribution, plot_background_emg_views, plot_hrs2_analysis,
    plot_actual_trial_timeline,
    # SNR analysis
    compute_snr_analysis, compute_mra_snr_analysis,
    plot_hrs2_trials, classify_trials, get_trial_window,
    detect_stim_onset, get_trial_context_window,
    detect_and_correct_failed_trials,
    # Post-hoc global windowing analysis
    analyze_global_background, run_threshold_sweep, plot_threshold_sweep,
    split_trials_by_polarity, plot_hm_ratio_summary, plot_hwave_regression,
    # Constants
    SAMPLE_RATE, BIN_DURATION_MS, BIN_SAMPLES, TRIAL_RECORD_MS,
    STIM_ONSET_THRESHOLD, STIM_END_THRESHOLD,
    build_merged_amp_groups,
)

print("Helpers loaded.")
print(f"  App constants: SAMPLE_RATE={SAMPLE_RATE} Hz | BIN={BIN_DURATION_MS} ms ({BIN_SAMPLES} samples) | TRIAL_RECORD={TRIAL_RECORD_MS} ms")
print(f"  Stim thresholds: onset >= {STIM_ONSET_THRESHOLD} V | end < {STIM_END_THRESHOLD} V")

Helpers loaded.
  App constants: SAMPLE_RATE=5000.0 Hz | BIN=50 ms (250 samples) | TRIAL_RECORD=100 ms
  Stim thresholds: onset >= 4.5 V | end < 1.9 V


# Section 2: HRS1 File — EMG Characterization (V1 only)

The cells in this section apply only to **V1** recordings where `.hrs1` contains
the EMG Characterization stage. They are automatically skipped for V2 recordings.

# Section 1b: Auto-Detect Recording Files

Set `recording_dir` to the path of your recording folder.  
The `.hrs1` and `.hrs2` files will be found automatically.

In [38]:
# C3_HRPILOT-17_BOOTH2_500US_6-22-26
# C4_HRPILOT-17_BOOTH2_500US_6-24-26
# C5_HRPILOT-17_BOOTH2_200US_6-24-26
# C6_HRPILOT-17_BOOTH2_200US_6-25-26
# C7_HRPILOT-17_BOOTH2_200US_6-26-26
# C8_HRPILOT-17_BOOTH2_200US_6-26-26


recording_dir         = "TEST4_HRPILOT-21_BOOTH2_500US_6-22-26"
recording_sample_rate = 15000  # Set to e.g. 5000.0 or 10000.0 to override; None = auto-detect from HRS1 header

hrs1_path, hrs2_path, hrs3_path = find_hrs_files(recording_dir)
_app_version = detect_app_version(recording_dir)
print(f"H-Reflex App V{_app_version}  |  "
      f"hrs1={os.path.basename(hrs1_path) if hrs1_path else chr(8211)}  "
      f"hrs2={os.path.basename(hrs2_path) if hrs2_path else chr(8211)}  "
      f"hrs3={os.path.basename(hrs3_path) if hrs3_path else chr(8211)}")


H-Reflex App V2  |  hrs1=TEST4_HRPILOT-21_BOOTH2_500US_6-22-26_20260622T133957.hrs1  hrs2=–  hrs3=–


In [39]:
# --- Load all data files (auto-detects V1 / V2 app) ---------------------------
cm_header  = cm_trials  = cm_emg_blocks  = None   # V2 Control Mode (None in V1)
dcp_header = dcp_trials = dcp_emg_blocks = None   # V2 Down Condition Pellet (None in V1)

if _app_version == 1:
    # V1: S1=EMG Characterization (.hrs1)  S2=MH Recruitment Curve (.hrs2)
    if hrs1_path:
        hrs1_header, hrs1_trials, hrs1_emg_blocks = read_hrs1(hrs1_path)
        print_hrs1_summary(hrs1_header, hrs1_trials, hrs1_emg_blocks, hrs1_path)
    else:
        print("No .hrs1 file found — EMG Characterization data unavailable.")
        hrs1_header, hrs1_trials, hrs1_emg_blocks = None, [], []
    if hrs2_path:
        hrs2_header, hrs2_trials, hrs2_emg_blocks = read_hrs2(hrs2_path)
        print_hrs2_summary(hrs2_header, hrs2_trials, hrs2_emg_blocks, hrs2_path)
    else:
        print("No .hrs2 file found — MH Recruitment data unavailable.")
        hrs2_header, hrs2_trials, hrs2_emg_blocks = None, [], []
else:
    # V2: S1=MH Recruitment (.hrs1, same binary as V1 .hrs2)
    #     S2=Control Mode (.hrs2)   S3=Down Condition Pellet (.hrs3)
    if hrs1_path:
        hrs2_header, hrs2_trials, hrs2_emg_blocks = read_hrs2(hrs1_path)
        hrs1_header = hrs2_header          # alias — no separate EMG char stage in V2
        hrs1_trials, hrs1_emg_blocks = [], []
        print_hrs2_summary(hrs2_header, hrs2_trials, hrs2_emg_blocks, hrs1_path)
    else:
        print("No .hrs1 file found — MH Recruitment data unavailable.")
        hrs1_header = hrs2_header = None
        hrs1_trials = hrs2_trials = []
        hrs1_emg_blocks = hrs2_emg_blocks = []
    if hrs2_path:
        cm_header, cm_trials, cm_emg_blocks = read_hrs2(hrs2_path)
        print(f"Control Mode:           {len(cm_trials)} trials")
    else:
        print("No .hrs2 file found — Control Mode data unavailable.")
    if hrs3_path:
        dcp_header, dcp_trials, dcp_emg_blocks = read_hrs3(hrs3_path)
        print(f"Down Condition Pellet:  {len(dcp_trials)} trials")
    else:
        print("No .hrs3 file found — Down Condition Pellet data unavailable.")


# V2 Control Mode-only: no Recruitment Curve stage (.hrs1 absent).
# Alias hrs2_trials <- cm_trials so all downstream analysis cells work unchanged.
if _app_version == 2 and not hrs2_trials and cm_trials:
    hrs2_header     = cm_header
    hrs2_trials     = cm_trials
    hrs2_emg_blocks = cm_emg_blocks
    hrs1_header     = hrs2_header
    print("Note: Control Mode trials used as primary analysis (no Recruitment Curve stage).")

# Ensure hrs1_header.sample_rate is always accessible for downstream cells
if hrs1_header is None:
    class _SampleRateStub:
        sample_rate = recording_sample_rate or 5000.0
    hrs1_header = _SampleRateStub()


=== HRS2 Header ===
  File:               TEST4_HRPILOT-21_BOOTH2_500US_6-22-26_20260622T133957.hrs1
  File version:       7  (v7: + digital_onset_sample_num/channel)
  Subject ID:         TEST4_HRPILOT-21_BOOTH2_500US_6-22-26
  Session start time: 2026-06-22 13:39:57.517318
  Stage name:         S1
  Stage description:  MH Recruitment Curve
  Stage type:         1

  Trials found:       837
  EMG data blocks:    199086
  Stim polarity:      534 normal, 303 reversed  <-- dual-polarity session
  Digital onsets:     0/837 trials have digital onset  <-- none detected

=== First EMG Block ===
  Channel names:  ['CH10', 'CH12', 'ADC1', 'ADC2', 'ADC3', 'ADC4', 'ADC5']
  Raw channels:   7 x 192 samples
  Diff samples:   192
No .hrs2 file found — Control Mode data unavailable.
No .hrs3 file found — Down Condition Pellet data unavailable.


In [40]:
# V1 EMG Characterization plots — skipped for V2 recordings or when .hrs1 is absent.
if not hrs1_trials:
    print('No EMG Characterization stage data — skipping V1 EMG char plots.')
else:
    # ---- Plot HRS1: Trial Grand Means (Session History) ----
    grand_means = [t.grand_mean for t in hrs1_trials]
    
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(range(len(grand_means)), grand_means, 'o', color='blue', markersize=6)
    ax.set_xlabel('Trial Number')
    ax.set_ylabel('Grand Mean (uV)')
    ax.set_title(f'HRS1 Session History - {hrs1_header.subject_id} ({hrs1_header.session_datetime:%Y-%m-%d %H:%M})')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"Grand mean range: [{min(grand_means):.2f}, {max(grand_means):.2f}] uV")
    print(f"Overall mean of grand means: {np.mean(grand_means):.2f} uV")
    
    # ---- Plot HRS1: Histogram of Grand Means ----
    gm = np.array(grand_means)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    hist_vals, hist_edges, _ = ax.hist(gm, bins=50, color='steelblue', edgecolor='black', alpha=0.8)
    #ax.axvline(hrs1_header.trial_initiation_uv_min, color='red', linestyle='--', linewidth=1.5, label=f'Init min = {hrs1_header.trial_initiation_uv_min}')
    #ax.axvline(hrs1_header.trial_initiation_uv_max, color='red', linestyle='--', linewidth=1.5, label=f'Init max = {hrs1_header.trial_initiation_uv_max}')
    
    q25, q50, q75 = np.percentile(gm, [25, 50, 75])
    ax.axvline(q25, color='orange', linestyle=':', linewidth=1.5, label=f'Q1 = {q25:.2f}')
    ax.axvline(q50, color='purple', linestyle='-.', linewidth=1.5, label=f'Median = {q50:.2f}')
    ax.axvline(q75, color='orange', linestyle=':', linewidth=1.5, label=f'Q3 = {q75:.2f}')
    
    ax.set_xlabel('Grand Mean Amplitude (uV)')
    ax.set_ylabel('Count')
    ax.set_title(f'HRS1 Histogram of Trial Grand Means - {hrs1_header.subject_id} (n={len(gm)})')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"Stats: min={gm.min():.2f}, Q1={q25:.2f}, median={q50:.2f}, Q3={q75:.2f}, max={gm.max():.2f}, std={gm.std():.2f}")
    
    # ---- Plot HRS1: Individual Trial Waveforms (first 6 trials) ----
    n_show = min(6, len(hrs1_trials))
    
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    axes = axes.flatten()
    
    for i in range(n_show):
        ax = axes[i]
        trial = hrs1_trials[i]
        sig = np.array(trial.monitored_signal)
        bins_arr = np.array(trial.bins)
        bin_sample_count = int((SAMPLE_RATE / 1000) * hrs1_header.bin_duration_ms)
    
        # Time axis in ms
        t_ms = np.arange(len(sig)) / SAMPLE_RATE * 1000.0
    
        # Plot monitored signal (abs EMG)
        ax.plot(t_ms, sig, color='black', linewidth=0.5, label='Monitored signal')
    
        # Plot bins
        bin_x = np.arange(len(bins_arr)) * hrs1_header.bin_duration_ms
        ax.plot(bin_x, bins_arr, color='red', linewidth=2, label='Bins')
    
        ax.set_title(f'Trial {i+1} | GM={trial.grand_mean:.2f} uV', fontsize=10)
        ax.set_xlabel('Time (ms)')
        ax.set_ylabel('Amplitude (uV)')
        ax.grid(True, alpha=0.3)
    
        if i == 0:
            ax.legend(fontsize=8)
    
    for j in range(n_show, len(axes)):
        axes[j].axis('off')
    
    fig.suptitle(f'HRS1 Trial Waveforms - {hrs1_header.subject_id}', fontsize=13)
    plt.tight_layout()
    plt.show()


No EMG Characterization stage data — skipping V1 EMG char plots.


# Section 3: Peri-Stimulus Trials — MH Recruitment Curve or Control Mode

For **V2** recordings `hrs2_trials` contains whichever stage was run:
- `.hrs1` present → MH Recruitment Curve trials
- `.hrs1` absent, `.hrs2` present → Control Mode trials (aliased automatically)

For **V1** recordings `hrs2_trials` contains the MH Recruitment Curve trials from `.hrs2`.

In [41]:
# Data already loaded in the cell above.
# hrs2_trials = MH Recruitment Curve trials (V1 or V2).
print(f"MH Recruitment trials : {len(hrs2_trials)}"
      + (f"  |  Control Mode: {len(cm_trials)}" if cm_trials  is not None else "")
      + (f"  |  DCP: {len(dcp_trials)}"         if dcp_trials is not None else ""))


MH Recruitment trials : 837


In [ ]:
# ── Stage Selector + Stimulation Intensity Histogram ──────────────────────────
# ACTIVE_STAGE controls which stage the Post-Hoc notebook (and _plot_trials below) uses.
# All viewer cells further down show a Stage dropdown when multiple stages are loaded.

_stage_map = {}
if hrs2_trials and (not cm_trials or hrs2_trials is not cm_trials):
    _stage_map['mh_recruitment'] = (hrs2_trials, hrs2_header, hrs2_emg_blocks,
                                     'MH Recruitment Curve (.hrs1)')
if cm_trials:
    _stage_map['control_mode']   = (cm_trials,   cm_header,   cm_emg_blocks,
                                     'Control Mode (.hrs2)')
if dcp_trials:
    _stage_map['dcp']            = (dcp_trials,  dcp_header,  dcp_emg_blocks,
                                     'Down Condition Pellet (.hrs3)')

ACTIVE_STAGE = next(iter(_stage_map))   # default: first available stage
# ACTIVE_STAGE = 'mh_recruitment'        # MH Recruitment Curve  (.hrs1)
# ACTIVE_STAGE = 'control_mode'          # Control Mode          (.hrs2)
# ACTIVE_STAGE = 'dcp'                   # Down Condition Pellet (.hrs3)

if ACTIVE_STAGE not in _stage_map:
    ACTIVE_STAGE = next(iter(_stage_map))
    print(f'Note: requested stage not available; defaulting to {ACTIVE_STAGE!r}')

_sel = _stage_map[ACTIVE_STAGE]
_plot_trials, _plot_header, _plot_emg_blocks = _sel[0], _sel[1], _sel[2]

print('Available stages:')
for _k, (_t, _h, _e, _lbl) in _stage_map.items():
    _mark = '  ◄ active' if _k == ACTIVE_STAGE else ''
    print(f'  {_k!r:<22} → {_lbl}  ({len(_t)} trials){_mark}')
print(f'\nActive : {_sel[3]}  ·  {len(_plot_trials)} trials')
print('(Change ACTIVE_STAGE above + re-run this cell only when switching stages for Post-Hoc analysis.)')

# ── Stimulation Intensity Histogram ───────────────────────────────────────
from ipywidgets import Dropdown, VBox, Output
from IPython.display import display as _disp

_hist_opts = [(lbl, sk) for sk, (_t, _h, _e, lbl) in _stage_map.items() if _t]
if len(_hist_opts) == 1:
    _sk, (_st, _sh, _se, _slbl) = next(iter(_stage_map.items()))
    print(f'\n── Histogram: {_slbl}  ({len(_st)} trials)')
    plot_amplitude_distribution(_st, _sh)
else:
    _hist_out  = Output()
    _hist_drop = Dropdown(options=_hist_opts, description='Stage:', layout={'width': '440px'})
    def _hist_show(change=None):
        _st, _sh, _se, _slbl = _stage_map[_hist_drop.value]
        with _hist_out:
            _hist_out.clear_output(wait=True)
            print(f'\n── Histogram: {_slbl}  ({len(_st)} trials)')
            plot_amplitude_distribution(_st, _sh)
    _hist_drop.observe(_hist_show, names='value')
    _disp(VBox([_hist_drop, _hist_out]))
    _hist_show()


In [ ]:
# ── Trial Timeline ─────────────────────────────────────────────────────────
# Shows trial number vs. actual wall-clock time and inter-trial interval (ITI)
# distribution for each recording stage.
from ipywidgets import Dropdown, VBox, Output
from IPython.display import display as _disp

_tl_opts = [(lbl, sk) for sk, (_t, _h, _e, lbl) in _stage_map.items() if _t]
if len(_tl_opts) == 1:
    _sk, (_st, _sh, _se, _slbl) = next(iter(_stage_map.items()))
    print(f'\n── Trial Timeline: {_slbl}  ({len(_st)} trials)')
    plot_actual_trial_timeline(_st, header=_sh)
else:
    _tl_out  = Output()
    _tl_drop = Dropdown(options=_tl_opts, description='Stage:', layout={'width': '440px'})
    def _tl_show(change=None):
        _st, _sh, _se, _slbl = _stage_map[_tl_drop.value]
        with _tl_out:
            _tl_out.clear_output(wait=True)
            print(f'\n── Trial Timeline: {_slbl}  ({len(_st)} trials)')
            plot_actual_trial_timeline(_st, header=_sh)
    _tl_drop.observe(_tl_show, names='value')
    _disp(VBox([_tl_drop, _tl_out]))
    _tl_show()


# Section 3a: Initiation Thresholds Summary

Displays the EMG amplitude thresholds and timing windows used to initiate trials in both stages:

- **S1 (EMG Characterization):** fixed thresholds stored in the file header â€” the grand mean of binned absolute EMG must fall between `trial_initiation_uv_min` and `trial_initiation_uv_max`.
- **S2 (MH Recruitment Curve):** per-trial thresholds stored with each trial â€” the app derives them from the S1 histogram percentiles, so they may vary if the experimenter re-ran the calculation.

In [43]:
# ---- Initiation Thresholds Summary ----

print("=" * 62)
print("  INITIATION THRESHOLDS SUMMARY")
print("=" * 62)

# --- S1: EMG Characterization (values stored in HRS1 header) ---
print("\nS1 EMG Characterization Stage")
if hasattr(hrs1_header, 'trial_initiation_uv_min'):
    print(f"  Lower bound : {hrs1_header.trial_initiation_uv_min:.2f} µV")
    print(f"  Upper bound : {hrs1_header.trial_initiation_uv_max:.2f} µV")
    print(f"  (A trial is accepted when the grand mean of binned |EMG| falls within this range)")
    print(f"  Monitoring window : {hrs1_header.trial_initiation_phase_min_ms}–{hrs1_header.trial_initiation_phase_max_ms} ms  |  Bin duration : {hrs1_header.bin_duration_ms} ms")
else:
    print("  (No EMG Characterization stage — V2 recording or V1 without .hrs1 file)")

# --- S2: MH Recruitment Curve (lower/upper bounds stored per trial in HRS2) ---
print("\nS2 MH Recruitment Curve Stage")
if _plot_header is None or len(_plot_trials) == 0:
    print("  No HRS2 file / no trials found.")
else:
    s2_mins = [t.min_initiation_threshold for t in _plot_trials]
    s2_maxs = [t.max_initiation_threshold for t in _plot_trials]
    all_same = (len(set(round(v, 4) for v in s2_mins)) == 1 and
                len(set(round(v, 4) for v in s2_maxs)) == 1)
    if all_same:
        print(f"  Lower bound : {s2_mins[0]:.4f} µV  (constant across all {len(_plot_trials)} trials)")
        print(f"  Upper bound : {s2_maxs[0]:.4f} µV  (constant across all {len(_plot_trials)} trials)")
    else:
        print(f"  Thresholds varied across {len(_plot_trials)} trials:")
        print(f"  {'Trial':>6}  {'Lower bound (µV)':>18}  {'Upper bound (µV)':>18}")
        for i, (mn, mx) in enumerate(zip(s2_mins, s2_maxs)):
            print(f"  {i + 1:>6}  {mn:>18.4f}  {mx:>18.4f}")

print("\n" + "=" * 62)

  INITIATION THRESHOLDS SUMMARY

S1 EMG Characterization Stage
  (No EMG Characterization stage — V2 recording or V1 without .hrs1 file)

S2 MH Recruitment Curve Stage
  Thresholds varied across 837 trials:
   Trial    Lower bound (µV)    Upper bound (µV)
       1              0.0000            600.0000
       2              0.0000            600.0000
       3              0.0000            100.0000
       4              0.0000            100.0000
       5              0.0000            100.0000
       6              0.0000            100.0000
       7              0.0000            100.0000
       8              0.0000            100.0000
       9              0.0000            100.0000
      10              0.0000            100.0000
      11              0.0000            100.0000
      12              0.0000            100.0000
      13              0.0000            100.0000
      14              0.0000            100.0000
      15              0.0000            100.0000
      16 

In [44]:
shared_ylim = (-1500, 1500)  # Set a common y-axis limit for all trial plots

In [45]:
#  Configuration 
PRE_PLOT_MS  = 2   # ms before stim onset to display
POST_PLOT_MS = 15  # ms after  stim onset to display
N_PER_PAGE   = 6   # trials shown per page (2 rows x 3 cols)

# M/H wave window constants (ms relative to stim onset)
M_WAVE_START_MS = 1.8 
M_WAVE_END_MS   = 5
H_WAVE_START_MS = 6
H_WAVE_END_MS   = 9

PRE_AVG_MS  = 2   # ms before stim onset
POST_AVG_MS = 15  # ms after  stim onset
N_PER_PAGE  = 6   # amplitude groups per page (2 rows x 3 cols)


In [46]:
'''# ---- Failed Trial Detector & Corrected Trial Windowing ----
# Classifies all trials for ADC-sync failures, realigns each failed trial to the
# true stim onset found via the first ADC pulse in the continuous context window,
# and plots original (gray) vs corrected (black) waveforms.
# Returns trial_report, failed, passed, realigned for use in later cells.
trial_report, failed, passed, realigned = detect_and_correct_failed_trials(
    _plot_trials, _plot_header, _plot_emg_blocks,
    pre_ms=PRE_PLOT_MS, post_ms=POST_PLOT_MS,
    m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
    h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
    sample_rate=recording_sample_rate or hrs1_header.sample_rate,
)'''

'# ---- Failed Trial Detector & Corrected Trial Windowing ----\n# Classifies all trials for ADC-sync failures, realigns each failed trial to the\n# true stim onset found via the first ADC pulse in the continuous context window,\n# and plots original (gray) vs corrected (black) waveforms.\n# Returns trial_report, failed, passed, realigned for use in later cells.\ntrial_report, failed, passed, realigned = detect_and_correct_failed_trials(\n    _plot_trials, _plot_header, _plot_emg_blocks,\n    pre_ms=PRE_PLOT_MS, post_ms=POST_PLOT_MS,\n    m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,\n    h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,\n    sample_rate=recording_sample_rate or hrs1_header.sample_rate,\n)'

In [47]:
# ── Optional: Merged Amplitude Group Analysis ─────────────────────────────
# Pool multiple stimulation intensities into one merged group.
# MERGED_GROUPS is a list of lists, e.g.:
#   [[0.12, 0.13], [0.15, 0.16, 0.17]]  →  two merged groups
# Leave as [] to use the default (unmerged) grouping.
MERGED_GROUPS = []

if MERGED_GROUPS:
    _hrs2_trials_plot = build_merged_amp_groups(_plot_trials, MERGED_GROUPS)
else:
    _hrs2_trials_plot = _plot_trials


In [ ]:

# ----- Failed Trial Detector  and Corrected Trial Windowing  Here----

# ── HRS2 Analysis: Interactive Averaged Waveforms + Recruitment Curve ─────
from ipywidgets import Dropdown, VBox, Output
from IPython.display import display as _disp

_ana_opts = [(lbl, sk) for sk, (_t, _h, _e, lbl) in _stage_map.items() if _t]
if len(_ana_opts) == 1:
    _sk, (_st, _sh, _se, _slbl) = next(iter(_stage_map.items()))
    _tp = build_merged_amp_groups(_st, MERGED_GROUPS) if MERGED_GROUPS else _st
    print(f'\n── Analysis: {_slbl}  ({len(_st)} trials)')
    plot_hrs2_analysis(
        _tp, _sh,
        pre_avg_ms=PRE_AVG_MS, post_avg_ms=POST_AVG_MS,
        n_per_page=N_PER_PAGE,
        m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
        h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
        sample_rate=recording_sample_rate or hrs1_header.sample_rate,
        emg_blocks=_se,
    )
else:
    _ana_out  = Output()
    _ana_drop = Dropdown(options=_ana_opts, description='Stage:', layout={'width': '440px'})
    def _ana_show(change=None):
        _st, _sh, _se, _slbl = _stage_map[_ana_drop.value]
        _tp = build_merged_amp_groups(_st, MERGED_GROUPS) if MERGED_GROUPS else _st
        with _ana_out:
            _ana_out.clear_output(wait=True)
            print(f'\n── Analysis: {_slbl}  ({len(_st)} trials)')
            plot_hrs2_analysis(
                _tp, _sh,
                pre_avg_ms=PRE_AVG_MS, post_avg_ms=POST_AVG_MS,
                n_per_page=N_PER_PAGE,
                m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
                h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
                sample_rate=recording_sample_rate or hrs1_header.sample_rate,
                emg_blocks=_se,
            )
    _ana_drop.observe(_ana_show, names='value')
    _disp(VBox([_ana_drop, _ana_out]))
    _ana_show()


In [ ]:
# ── HRS2 Trial Viewer: Interactive Per-Trial Grid + Zoom ─────────────────
from ipywidgets import Dropdown, VBox, Output
from IPython.display import display as _disp

_trv_opts = [(lbl, sk) for sk, (_t, _h, _e, lbl) in _stage_map.items() if _t]
if len(_trv_opts) == 1:
    _sk, (_st, _sh, _se, _slbl) = next(iter(_stage_map.items()))
    _tp = build_merged_amp_groups(_st, MERGED_GROUPS) if MERGED_GROUPS else _st
    print(f'\n── Trial Viewer: {_slbl}  ({len(_st)} trials)')
    plot_hrs2_trials(
        _tp, _sh,
        pre_plot_ms=PRE_PLOT_MS, post_plot_ms=POST_PLOT_MS,
        n_per_page=N_PER_PAGE,
        m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
        h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
        sample_rate=recording_sample_rate or hrs1_header.sample_rate,
        emg_blocks=_se,
    )
else:
    _trv_out  = Output()
    _trv_drop = Dropdown(options=_trv_opts, description='Stage:', layout={'width': '440px'})
    def _trv_show(change=None):
        _st, _sh, _se, _slbl = _stage_map[_trv_drop.value]
        _tp = build_merged_amp_groups(_st, MERGED_GROUPS) if MERGED_GROUPS else _st
        with _trv_out:
            _trv_out.clear_output(wait=True)
            print(f'\n── Trial Viewer: {_slbl}  ({len(_st)} trials)')
            plot_hrs2_trials(
                _tp, _sh,
                pre_plot_ms=PRE_PLOT_MS, post_plot_ms=POST_PLOT_MS,
                n_per_page=N_PER_PAGE,
                m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
                h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
                sample_rate=recording_sample_rate or hrs1_header.sample_rate,
                emg_blocks=_se,
            )
    _trv_drop.observe(_trv_show, names='value')
    _disp(VBox([_trv_drop, _trv_out]))
    _trv_show()


# Section 3c: H:M Ratio Summary

Box plot and histogram of H:M ratio for each stimulation polarity group.
If both normal and reversed polarities were used in this session, each group is analysed separately.

In [ ]:
# ── Stim Polarity Analysis ───────────────────────────────────────────────
from ipywidgets import Dropdown, VBox, Output
from IPython.display import display as _disp

_pol_opts = [(lbl, sk) for sk, (_t, _h, _e, lbl) in _stage_map.items() if _t]
if len(_pol_opts) == 1:
    _sk, (_st, _sh, _se, _slbl) = next(iter(_stage_map.items()))
    print(f'\n── Polarity / H:M Ratio: {_slbl}  ({len(_st)} trials)')
    trials_by_polarity = split_trials_by_polarity(_st)
    plot_hm_ratio_summary(
        trials_by_polarity, _sh,
        m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
        h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
        sample_rate=recording_sample_rate or hrs1_header.sample_rate,
        pre_ms=PRE_PLOT_MS, post_ms=POST_PLOT_MS,
    )
    plot_hwave_regression(
        _st, _se,
        m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
        h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
        sample_rate=recording_sample_rate or hrs1_header.sample_rate,
        pre_ms=PRE_PLOT_MS, post_ms=POST_PLOT_MS,
    )
else:
    _pol_out  = Output()
    _pol_drop = Dropdown(options=_pol_opts, description='Stage:', layout={'width': '440px'})
    def _pol_show(change=None):
        _st, _sh, _se, _slbl = _stage_map[_pol_drop.value]
        with _pol_out:
            _pol_out.clear_output(wait=True)
            print(f'\n── Polarity / H:M Ratio: {_slbl}  ({len(_st)} trials)')
            trials_by_polarity = split_trials_by_polarity(_st)
            plot_hm_ratio_summary(
                trials_by_polarity, _sh,
                m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
                h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
                sample_rate=recording_sample_rate or hrs1_header.sample_rate,
                pre_ms=PRE_PLOT_MS, post_ms=POST_PLOT_MS,
            )
            plot_hwave_regression(
                _st, _se,
                m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
                h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
                sample_rate=recording_sample_rate or hrs1_header.sample_rate,
                pre_ms=PRE_PLOT_MS, post_ms=POST_PLOT_MS,
            )
    _pol_drop.observe(_pol_show, names='value')
    _disp(VBox([_pol_drop, _pol_out]))
    _pol_show()


In [ ]:
# ── Averaged Waveforms Grouped by Stimulation Amplitude ──────────────────
from collections import defaultdict
from ipywidgets import Dropdown, VBox, Output
from IPython.display import display as _disp

PRE_AVG_MS  = 2   # ms before stim onset
POST_AVG_MS = 15  # ms after  stim onset
M_WAVE_START_MS_AVG = 2.5
M_WAVE_END_MS_AVG   = 4.0
H_WAVE_START_MS_AVG = 6.0
H_WAVE_END_MS_AVG   = 10.0

def _plot_avg_waveforms(_st, _se, _slbl):
    print(f'\n── Averaged Waveforms: {_slbl}  ({len(_st)} trials)')
    groups = defaultdict(list)
    for trial in _st:
        key = round(trial.stimulation_amplitude_ma, 2)
        t_ms_win, emg_win, _, stim_end = get_trial_window(trial, PRE_AVG_MS, POST_AVG_MS)
        groups[key].append((t_ms_win, emg_win, stim_end))
    _all_emg = np.concatenate([emg for windows in groups.values() for _, emg, _ in windows])
    _lo, _hi = float(np.nanmin(_all_emg)), float(np.nanmax(_all_emg))
    _pad = max(0.08 * (_hi - _lo), 1.0)
    shared_ylim = (_lo - _pad, _hi + _pad)
    _text_offset = 0.05 * (shared_ylim[1] - shared_ylim[0])
    for amp in sorted(groups.keys()):
        windows = groups[amp]
        t_ref = windows[0][0]
        n_pts = len(t_ref)
        padded = np.full((len(windows), n_pts), np.nan)
        for k, (_, emg, _se2) in enumerate(windows):
            n = min(len(emg), n_pts)
            padded[k, :n] = emg[:n]
        avg = np.nanmean(padded, axis=0)
        stim_ends = [se for _, _, se in windows if se is not None]
        mean_stim_end = float(np.mean(stim_ends)) if stim_ends else 0.5
        plt.figure(figsize=(10, 5))
        for row in padded:
            plt.plot(t_ref, row, color='red', alpha=0.6)
        plt.plot(t_ref, avg, color='black', linewidth=2, label='Average EMG')
        plt.axvspan(0, mean_stim_end, color='red', alpha=0.20,
                    label=f'Stim period (0–{mean_stim_end:.1f} ms)')
        plt.axvline(x=0, color='red', linestyle='--', label='Stim onset')
        plt.axvline(x=mean_stim_end, color='red', linestyle='--',
                    label=f'Stim end (~{mean_stim_end:.1f} ms)')
        plt.axvspan(M_WAVE_START_MS_AVG, M_WAVE_END_MS_AVG, color='blue',  alpha=0.3)
        plt.axvspan(H_WAVE_START_MS_AVG, H_WAVE_END_MS_AVG, color='green', alpha=0.3)
        m_mask = (t_ref >= M_WAVE_START_MS_AVG) & (t_ref <= M_WAVE_END_MS_AVG)
        m_t, m_emg = t_ref[m_mask], avg[m_mask]
        m_pi = np.argmax(m_emg)
        h_mask = (t_ref >= H_WAVE_START_MS_AVG) & (t_ref <= H_WAVE_END_MS_AVG)
        h_t, h_emg = t_ref[h_mask], avg[h_mask]
        h_pi = np.argmax(h_emg)
        plt.axvline(x=m_t[m_pi], color='blue', linestyle=':', linewidth=2,
                    label=f'M-Wave Peak: {m_emg[m_pi]:.1f} µV at {m_t[m_pi]:.2f} ms')
        plt.axvline(x=h_t[h_pi], color='green', linestyle=':', linewidth=2,
                    label=f'H-Wave Peak: {h_emg[h_pi]:.1f} µV at {h_t[h_pi]:.2f} ms')
        plt.text(m_t[m_pi], m_emg[m_pi] + _text_offset, f'{m_emg[m_pi]:.1f} µV',
                 color='blue', fontsize=9, ha='center')
        plt.text(h_t[h_pi], h_emg[h_pi] + _text_offset, f'{h_emg[h_pi]:.1f} µV',
                 color='green', fontsize=9, ha='center')
        plt.title(f'Averaged Waveforms [{_slbl}] (n={len(windows)}) | Stim Amp: {amp:.2f} mA', fontsize=20)
        plt.xlabel('Time (ms)')
        plt.ylabel('Amplitude (µV)')
        plt.grid(True)
        plt.legend(fontsize=7, loc='upper left', frameon=True, framealpha=0.8)
        plt.ylim(-1000, 1500)
        plt.xticks(np.arange(int(np.floor(t_ref[0])), int(np.ceil(t_ref[-1])) + 1, 1))
        plt.tight_layout()
        plt.show()

_avg_opts = [(lbl, sk) for sk, (_t, _h, _e, lbl) in _stage_map.items() if _t]
if len(_avg_opts) == 1:
    _sk, (_st, _sh, _se, _slbl) = next(iter(_stage_map.items()))
    _plot_avg_waveforms(_st, _se, _slbl)
else:
    _avg_out  = Output()
    _avg_drop = Dropdown(options=_avg_opts, description='Stage:', layout={'width': '440px'})
    def _avg_show(change=None):
        _st, _sh, _se, _slbl = _stage_map[_avg_drop.value]
        with _avg_out:
            _avg_out.clear_output(wait=True)
            _plot_avg_waveforms(_st, _se, _slbl)
    _avg_drop.observe(_avg_show, names='value')
    _disp(VBox([_avg_drop, _avg_out]))
    _avg_show()


# Section 4: Polarity Test CORRECT vs FLIPPED Electrode Orientation

This section loads both polarity test recordings and compares them side-by-side.

The app applies `|ADC sync|` (absolute value) on the ADC sync line so that stim-onset detection works regardless of electrode orientation.  
The raw EMG differential signal (`CH8 âˆ’ CH7`) will appear **inverted** when the electrodes are swapped.

- **`HR-08-POLARITY_TEST_CORRECT`** electrodes attached with standard orientation
- **`HR-08-POLARITY_TEST_FLIPPED_`** electrodes attached with reversed orientation

In [52]:
'''# ---- Load both polarity recordings ----
config_A_dir = "HRPILOT-16_CONFIGA_POSTHOC_WINDOWING"
config_B_dir = "HRPILOT-16_CONFIGB_POSTHOC_WINDOWING"

_hrs1_correct_path, _hrs2_correct_path = find_hrs_files(config_A_dir)
_hrs1_flipped_path, _hrs2_flipped_path = find_hrs_files(config_B_dir)

_h2c, _t2c, _e2c = read_hrs2(_hrs2_correct_path)
_h2f, _t2f, _e2f = read_hrs2(_hrs2_flipped_path)

print(f"CORRECT : {os.path.basename(_hrs2_correct_path)}")
print(f"  file_version={_h2c.file_version}  trials={len(_t2c)}  emg_blocks={len(_e2c)}")
print(f"  channels: {_e2c[0].channel_names if _e2c else 'n/a'}")
print()
print(f"FLIPPED : {os.path.basename(_hrs2_flipped_path)}")
print(f"  file_version={_h2f.file_version}  trials={len(_t2f)}  emg_blocks={len(_e2f)}")
print(f"  channels: {_e2f[0].channel_names if _e2f else 'n/a'}")

# Find amplitudes present in both recordings
_amps_c = set(round(t.stimulation_amplitude_ma, 2) for t in _t2c)
_amps_f = set(round(t.stimulation_amplitude_ma, 2) for t in _t2f)
_shared_amps = sorted(_amps_c & _amps_f)
print(f"\nShared stim amplitudes ({len(_shared_amps)}): {_shared_amps}")

# ---- Polarity Comparison: Interactive Paged Viewer ----
# 2x2 grid per page: 2 stimulus amplitudes per page.
# Top row = CORRECT polarity, bottom row = FLIPPED polarity.
# M- and H-wave windows are shaded on every panel.
# Requires an interactive Jupyter kernel (ipywidgets).

from collections import defaultdict
from ipywidgets import Button, Output, HBox, VBox, Dropdown, Label, Checkbox, ToggleButton, FloatText
from matplotlib.patches import Patch as _Patch

_PRE_COMP_MS  = 2.5
_POST_COMP_MS = 12.5

'''M_WAVE_START_MS = 2.5
M_WAVE_END_MS   = 4.0
H_WAVE_START_MS = 6.0
H_WAVE_END_MS   = 10.0'''

def _build_avg_groups(trials, pre_ms, post_ms):
    groups = defaultdict(list)
    for tr in trials:
        key = round(tr.stimulation_amplitude_ma, 2)
        t_ms, emg, _, _, _ = get_trial_window(tr, pre_ms, post_ms)
        groups[key].append((t_ms, emg))
    return groups

_grp_c = _build_avg_groups(_t2c, _PRE_COMP_MS, _POST_COMP_MS)
_grp_f = _build_avg_groups(_t2f, _PRE_COMP_MS, _POST_COMP_MS)

# 2 amplitudes per page → 4 subplots (2 cols × 2 rows)
_N_AMPS_PER_PAGE = 2
_pol_pages = [_shared_amps[i:i+_N_AMPS_PER_PAGE]
              for i in range(0, len(_shared_amps), _N_AMPS_PER_PAGE)]
_pol_current_page = {'idx': 0}

# Shared y-limits across all polarity traces
_pol_all_emg = np.concatenate([
    emg
    for grp in [_grp_c, _grp_f]
    for windows in grp.values()
    for _, emg in windows
])
_pol_lo, _pol_hi = float(np.nanmin(_pol_all_emg)), float(np.nanmax(_pol_all_emg))
_pol_pad = max(0.08 * (_pol_hi - _pol_lo), 1.0)
_pol_ylim = (_pol_lo - _pol_pad, _pol_hi + _pol_pad)

_show_abs = {'val': False}
_ylim_auto = {'val': True}
_ylim_man = {'lo': _pol_ylim[0], 'hi': _pol_ylim[1]}

_pol_out = Output()

def _get_ylim():
    if _ylim_auto['val']:
        arrays = [_pol_all_emg]
        if _show_abs['val']:
            arrays.append(np.abs(_pol_all_emg))
        _all = np.concatenate([a.ravel() for a in arrays])
        _all = _all[~np.isnan(_all)]
        if len(_all) == 0:
            return (-1500.0, 1500.0)
        _lo, _hi = float(np.nanmin(_all)), float(np.nanmax(_all))
        _pad = max(0.08 * (_hi - _lo), 1.0)
        return (_lo - _pad, _hi + _pad)
    return (_ylim_man['lo'], _ylim_man['hi'])


def _draw_pol_ax(ax, grp, amp, label, color):
    """Render averaged waveforms for one polarity/amplitude into ax."""
    windows = grp.get(amp, [])
    legend_h, legend_l = [], []

    if windows:
        t_ref = windows[0][0]
        n_pts = len(t_ref)
        padded = np.full((len(windows), n_pts), np.nan)
        for k, (_, emg) in enumerate(windows):
            n = min(len(emg), n_pts)
            padded[k, :n] = emg[:n]
        avg = np.nanmean(padded, axis=0)
        for row_data in padded:
            ax.plot(t_ref, row_data, color=color, alpha=0.25, linewidth=0.5)
        h_avg, = ax.plot(t_ref, avg, color=color, linewidth=2.0)
        legend_h.append(h_avg)
        legend_l.append(f'{label} avg (n={len(windows)})')

        if _show_abs['val']:
            abs_padded = np.abs(padded)
            avg_abs = np.nanmean(abs_padded, axis=0)
            for row_data in abs_padded:
                ax.plot(t_ref, row_data, color='gray', alpha=0.15, linewidth=0.5)
            h_abs, = ax.plot(t_ref, avg_abs, color='gray', linewidth=1.2)
            legend_h.append(h_abs)
            legend_l.append('|EMG| avg')

    # M/H wave region overlays
    ax.axvspan(M_WAVE_START_MS, M_WAVE_END_MS, color='blue',  alpha=0.12, zorder=0)
    ax.axvspan(H_WAVE_START_MS, H_WAVE_END_MS, color='green', alpha=0.12, zorder=0)
    legend_h += [_Patch(facecolor='blue', alpha=0.4), _Patch(facecolor='green', alpha=0.4)]
    legend_l += ['M-wave (2.5–4 ms)', 'H-wave (6–10 ms)']

    h_onset = ax.axvline(0, color='blue', linestyle='--', linewidth=1.0)
    legend_h.append(h_onset)
    legend_l.append('Stim onset')

    ax.axhline(0, color='gray', linestyle='-', linewidth=0.5, alpha=0.5)
    ax.set_xlim(-_PRE_COMP_MS, _POST_COMP_MS)
    ax.set_xlabel('Time (ms)', fontsize=8)
    ax.set_ylabel(f'{label}\nEMG (µV)', fontsize=8)
    ax.set_ylim(_get_ylim())
    ax.tick_params(labelsize=7)
    ax.grid(True, alpha=0.3)
    ax.legend(legend_h, legend_l, fontsize=7, loc='upper right')

def _plot_pol_page(page_idx):
    with _pol_out:
        _pol_out.clear_output(wait=True)
        amps_on_page = _pol_pages[page_idx]
        n_cols = len(amps_on_page)

        fig, axes = plt.subplots(2, n_cols, figsize=(6 * n_cols, 8), sharey='row')
        # Normalize axes to always be 2D array
        if n_cols == 1:
            axes = np.array([[axes[0]], [axes[1]]])
        else:
            axes = np.array(axes)

        for col, amp in enumerate(amps_on_page):
            for row_idx, (grp, label, color) in enumerate([
                    (_grp_c, 'CORRECT', 'black'),
                    (_grp_f, 'FLIPPED', 'firebrick'),
            ]):
                ax = axes[row_idx, col]
                _draw_pol_ax(ax, grp, amp, label, color)
                if row_idx == 0:
                    ax.set_title(f'{amp:.2f} mA', fontsize=10)

        fig.suptitle(
            f'Polarity Comparison — CORRECT vs FLIPPED\n'
            f'({_h2c.subject_id}  vs  {_h2f.subject_id})  '
            f'(Page {page_idx + 1}/{len(_pol_pages)})',
            fontsize=12
        )
        plt.tight_layout()
        plt.show()

def _pol_on_prev(b):
    if _pol_current_page['idx'] > 0:
        _pol_current_page['idx'] -= 1
        _pol_page_drop.value = _pol_current_page['idx']
        _plot_pol_page(_pol_current_page['idx'])

def _pol_on_next(b):
    if _pol_current_page['idx'] < len(_pol_pages) - 1:
        _pol_current_page['idx'] += 1
        _pol_page_drop.value = _pol_current_page['idx']
        _plot_pol_page(_pol_current_page['idx'])

def _pol_on_page_change(change):
    if change['name'] == 'value' and change['new'] != _pol_current_page['idx']:
        _pol_current_page['idx'] = change['new']
        _plot_pol_page(_pol_current_page['idx'])

_pol_prev_btn  = Button(description='Prev', button_style='')
_pol_next_btn  = Button(description='Next', button_style='primary')
_pol_page_drop = Dropdown(
    options=[(f'Page {i+1}', i) for i in range(len(_pol_pages))],
    description='Page:', layout={'width': '130px'}
)

_pol_prev_btn.on_click(_pol_on_prev)
_pol_next_btn.on_click(_pol_on_next)
_pol_page_drop.observe(_pol_on_page_change, names='value')

def _on_abs_change(change):
    if change['name'] == 'value':
        _show_abs['val'] = change['new']
        if _ylim_auto['val']:
            _pol_ymin.value, _pol_ymax.value = _get_ylim()
        _plot_pol_page(_pol_current_page['idx'])

def _on_auto_toggle(change):
    if change['name'] == 'value':
        _ylim_auto['val'] = change['new']
        _pol_ymin.disabled = _ylim_auto['val']
        _pol_ymax.disabled = _ylim_auto['val']
        if _ylim_auto['val']:
            _pol_ymin.value, _pol_ymax.value = _get_ylim()
        _plot_pol_page(_pol_current_page['idx'])

def _on_ymin_change(change):
    if change['name'] == 'value':
        _ylim_man['lo'] = change['new']
        if not _ylim_auto['val']:
            _plot_pol_page(_pol_current_page['idx'])

def _on_ymax_change(change):
    if change['name'] == 'value':
        _ylim_man['hi'] = change['new']
        if not _ylim_auto['val']:
            _plot_pol_page(_pol_current_page['idx'])

_pol_abs_cb = Checkbox(value=False, description='Show |EMG|', indent=False, layout={'width': '130px'})
_pol_auto_toggle = ToggleButton(
    value=True, description='Auto y-scale', button_style='success',
    tooltip='Auto-scale shared y-axis from visible signals'
)
_pol_ymin = FloatText(value=_pol_ylim[0], description='Y min:', disabled=True, layout={'width': '145px'})
_pol_ymax = FloatText(value=_pol_ylim[1], description='Y max:', disabled=True, layout={'width': '145px'})

_pol_abs_cb.observe(_on_abs_change, names='value')
_pol_auto_toggle.observe(_on_auto_toggle, names='value')
_pol_ymin.observe(_on_ymin_change, names='value')
_pol_ymax.observe(_on_ymax_change, names='value')

_pol_controls = VBox([
    HBox([_pol_prev_btn, _pol_next_btn, _pol_page_drop, _pol_abs_cb]),
    HBox([_pol_auto_toggle, _pol_ymin, _pol_ymax]),
])

print(f"{len(_shared_amps)} shared amplitudes across {len(_pol_pages)} page(s), 2 per page.")
print("Top row: CORRECT polarity  |  Bottom row: FLIPPED polarity")
print("Note: FLIPPED traces should be the vertical mirror image of CORRECT traces.")
display(VBox([_pol_controls, _pol_out]))
_plot_pol_page(0)
'''

SyntaxError: invalid syntax (3288642172.py, line 38)

In [ ]:
'''# ---- Load polarity recordings ----
config_A_dir = "HRPILOT-16_CONFIGA_POSTHOC_WINDOWING"
config_B_dir = "HRPILOT-16_CONFIGB_POSTHOC_WINDOWING"
config_AB_dir = "HRPILOT-16_CONFIGAB_POSTHOC_WINDOWING"

_hrs1_correct_path, _hrs2_correct_path = find_hrs_files(config_A_dir)
_hrs1_flipped_path, _hrs2_flipped_path = find_hrs_files(config_B_dir)
_hrs1_ab_path, _hrs2_ab_path = find_hrs_files(config_AB_dir)

_h2c, _t2c, _e2c = read_hrs2(_hrs2_correct_path)
_h2f, _t2f, _e2f = read_hrs2(_hrs2_flipped_path)
_h2ab, _t2ab, _e2ab = read_hrs2(_hrs2_ab_path)

print(f"CORRECT : {os.path.basename(_hrs2_correct_path)}")
print(f"  file_version={_h2c.file_version}  trials={len(_t2c)}  emg_blocks={len(_e2c)}")
print(f"  channels: {_e2c[0].channel_names if _e2c else 'n/a'}")
print()
print(f"FLIPPED : {os.path.basename(_hrs2_flipped_path)}")
print(f"  file_version={_h2f.file_version}  trials={len(_t2f)}  emg_blocks={len(_e2f)}")
print(f"  channels: {_e2f[0].channel_names if _e2f else 'n/a'}")
print()
print(f"CONFIGAB: {os.path.basename(_hrs2_ab_path)}")
print(f"  file_version={_h2ab.file_version}  trials={len(_t2ab)}  emg_blocks={len(_e2ab)}")
print(f"  channels: {_e2ab[0].channel_names if _e2ab else 'n/a'}")

# Find amplitudes present in all recordings
_amps_c = set(round(t.stimulation_amplitude_ma, 2) for t in _t2c)
_amps_f = set(round(t.stimulation_amplitude_ma, 2) for t in _t2f)
_amps_ab = set(round(t.stimulation_amplitude_ma, 2) for t in _t2ab)
_shared_amps = sorted(_amps_c & _amps_f & _amps_ab)
print(f"\nShared stim amplitudes ({len(_shared_amps)}): {_shared_amps}")

# ---- Polarity Comparison: Interactive Paged Viewer ----
# 3x2 grid per page: 2 stimulus amplitudes per page.
# Top row = CORRECT polarity, middle row = FLIPPED polarity, bottom row = CONFIGAB polarity.
# M- and H-wave windows are shaded on every panel.
# Requires an interactive Jupyter kernel (ipywidgets).

from collections import defaultdict
from ipywidgets import Button, Output, HBox, VBox, Dropdown, Label, Checkbox, ToggleButton, FloatText
from matplotlib.patches import Patch as _Patch

_PRE_COMP_MS  = 5.0
_POST_COMP_MS = 20.0

'''M_WAVE_START_MS = 2.5
M_WAVE_END_MS   = 4.0
H_WAVE_START_MS = 6.0
H_WAVE_END_MS   = 10.0'''

def _build_avg_groups(trials, pre_ms, post_ms):
    groups = defaultdict(list)
    for tr in trials:
        key = round(tr.stimulation_amplitude_ma, 2)
        t_ms, emg, _, _, _ = get_trial_window(tr, pre_ms, post_ms)
        groups[key].append((t_ms, emg))
    return groups

_grp_c = _build_avg_groups(_t2c, _PRE_COMP_MS, _POST_COMP_MS)
_grp_f = _build_avg_groups(_t2f, _PRE_COMP_MS, _POST_COMP_MS)
_grp_ab = _build_avg_groups(_t2ab, _PRE_COMP_MS, _POST_COMP_MS)

# 2 amplitudes per page → 6 subplots (3 cols × 2 rows)
_N_AMPS_PER_PAGE = 2
_pol_pages = [_shared_amps[i:i+_N_AMPS_PER_PAGE]
              for i in range(0, len(_shared_amps), _N_AMPS_PER_PAGE)]
_pol_current_page = {'idx': 0}

# Shared y-limits across all polarity traces
_pol_all_emg = np.concatenate([
    emg
    for grp in [_grp_c, _grp_f, _grp_ab]
    for windows in grp.values()
    for _, emg in windows
])
_pol_lo, _pol_hi = float(np.nanmin(_pol_all_emg)), float(np.nanmax(_pol_all_emg))
_pol_pad = max(0.08 * (_pol_hi - _pol_lo), 1.0)
_pol_ylim = (_pol_lo - _pol_pad, _pol_hi + _pol_pad)

_show_abs = {'val': False}
_ylim_auto = {'val': True}
_ylim_man = {'lo': _pol_ylim[0], 'hi': _pol_ylim[1]}

_pol_out = Output()

def _get_ylim():
    if _ylim_auto['val']:
        arrays = [_pol_all_emg]
        if _show_abs['val']:
            arrays.append(np.abs(_pol_all_emg))
        _all = np.concatenate([a.ravel() for a in arrays])
        _all = _all[~np.isnan(_all)]
        if len(_all) == 0:
            return (-1500.0, 1500.0)
        _lo, _hi = float(np.nanmin(_all)), float(np.nanmax(_all))
        _pad = max(0.08 * (_hi - _lo), 1.0)
        return (_lo - _pad, _hi + _pad)
    return (_ylim_man['lo'], _ylim_man['hi'])


def _draw_pol_ax(ax, grp, amp, label, color):
    """Render averaged waveforms for one polarity/amplitude into ax."""
    windows = grp.get(amp, [])
    legend_h, legend_l = [], []

    if windows:
        t_ref = windows[0][0]
        n_pts = len(t_ref)
        padded = np.full((len(windows), n_pts), np.nan)
        for k, (_, emg) in enumerate(windows):
            n = min(len(emg), n_pts)
            padded[k, :n] = emg[:n]
        avg = np.nanmean(padded, axis=0)
        for row_data in padded:
            ax.plot(t_ref, row_data, color=color, alpha=0.25, linewidth=0.5)
        h_avg, = ax.plot(t_ref, avg, color=color, linewidth=2.0)
        legend_h.append(h_avg)
        legend_l.append(f'{label} avg (n={len(windows)})')

        if _show_abs['val']:
            abs_padded = np.abs(padded)
            avg_abs = np.nanmean(abs_padded, axis=0)
            for row_data in abs_padded:
                ax.plot(t_ref, row_data, color='gray', alpha=0.15, linewidth=0.5)
            h_abs, = ax.plot(t_ref, avg_abs, color='gray', linewidth=1.2)
            legend_h.append(h_abs)
            legend_l.append('|EMG| avg')

    # M/H wave region overlays
    ax.axvspan(M_WAVE_START_MS, M_WAVE_END_MS, color='blue',  alpha=0.12, zorder=0)
    ax.axvspan(H_WAVE_START_MS, H_WAVE_END_MS, color='green', alpha=0.12, zorder=0)
    legend_h += [_Patch(facecolor='blue', alpha=0.4), _Patch(facecolor='green', alpha=0.4)]
    legend_l += ['M-wave (2.5–4 ms)', 'H-wave (6–10 ms)']

    h_onset = ax.axvline(0, color='blue', linestyle='--', linewidth=1.0)
    legend_h.append(h_onset)
    legend_l.append('Stim onset')

    ax.axhline(0, color='gray', linestyle='-', linewidth=0.5, alpha=0.5)
    ax.set_xlim(-_PRE_COMP_MS, _POST_COMP_MS)
    ax.set_xlabel('Time (ms)', fontsize=8)
    ax.set_ylabel(f'{label}\nEMG (µV)', fontsize=8)
    ax.set_ylim(_get_ylim())
    ax.tick_params(labelsize=7)
    ax.grid(True, alpha=0.3)
    ax.legend(legend_h, legend_l, fontsize=7, loc='upper right')

def _plot_pol_page(page_idx):
    with _pol_out:
        _pol_out.clear_output(wait=True)
        amps_on_page = _pol_pages[page_idx]
        n_amps = len(amps_on_page)
        n_configs = 3

        fig, axes = plt.subplots(n_amps, n_configs, figsize=(7 * n_configs, 5 * n_amps), sharey='col')
        # Normalize axes to always be 2D array
        if n_amps == 1:
            axes = axes[np.newaxis, :]
        elif n_configs == 1:
            axes = axes[:, np.newaxis]
        else:
            axes = np.array(axes)

        for row_idx, amp in enumerate(amps_on_page):
            for col_idx, (grp, label, color) in enumerate([
                    (_grp_c, 'CONFIGA', 'black'),
                    (_grp_f, 'CONFIGB', 'firebrick'),
                    (_grp_ab, 'CONFIGAB', 'blue'),
            ]):
                ax = axes[row_idx, col_idx]
                _draw_pol_ax(ax, grp, amp, label, color)
                if row_idx == 0:
                    ax.set_title(f'{label}', fontsize=11)
                if col_idx == 0:
                    ax.set_ylabel(f'{amp:.2f} mA {ax.get_ylabel()}', fontsize=20)

        fig.suptitle(
            f'Polarity Comparison — CORRECT vs FLIPPED vs CONFIGAB'
            f'({_h2c.subject_id}  vs  {_h2f.subject_id}  vs  {_h2ab.subject_id})  '
            f'(Page {page_idx + 1}/{len(_pol_pages)})',
            fontsize=12
        )
        plt.tight_layout()
        plt.show()
def _pol_on_prev(b):
    if _pol_current_page['idx'] > 0:
        _pol_current_page['idx'] -= 1
        _pol_page_drop.value = _pol_current_page['idx']
        _plot_pol_page(_pol_current_page['idx'])

def _pol_on_next(b):
    if _pol_current_page['idx'] < len(_pol_pages) - 1:
        _pol_current_page['idx'] += 1
        _pol_page_drop.value = _pol_current_page['idx']
        _plot_pol_page(_pol_current_page['idx'])

def _pol_on_page_change(change):
    if change['name'] == 'value' and change['new'] != _pol_current_page['idx']:
        _pol_current_page['idx'] = change['new']
        _plot_pol_page(_pol_current_page['idx'])

_pol_prev_btn  = Button(description='Prev', button_style='')
_pol_next_btn  = Button(description='Next', button_style='primary')
_pol_page_drop = Dropdown(
    options=[(f'Page {i+1}', i) for i in range(len(_pol_pages))],
    description='Page:', layout={'width': '130px'}
)

_pol_prev_btn.on_click(_pol_on_prev)
_pol_next_btn.on_click(_pol_on_next)
_pol_page_drop.observe(_pol_on_page_change, names='value')

def _on_abs_change(change):
    if change['name'] == 'value':
        _show_abs['val'] = change['new']
        if _ylim_auto['val']:
            _pol_ymin.value, _pol_ymax.value = _get_ylim()
        _plot_pol_page(_pol_current_page['idx'])

def _on_auto_toggle(change):
    if change['name'] == 'value':
        _ylim_auto['val'] = change['new']
        _pol_ymin.disabled = _ylim_auto['val']
        _pol_ymax.disabled = _ylim_auto['val']
        if _ylim_auto['val']:
            _pol_ymin.value, _pol_ymax.value = _get_ylim()
        _plot_pol_page(_pol_current_page['idx'])

def _on_ymin_change(change):
    if change['name'] == 'value':
        _ylim_man['lo'] = change['new']
        if not _ylim_auto['val']:
            _plot_pol_page(_pol_current_page['idx'])

def _on_ymax_change(change):
    if change['name'] == 'value':
        _ylim_man['hi'] = change['new']
        if not _ylim_auto['val']:
            _plot_pol_page(_pol_current_page['idx'])

_pol_abs_cb = Checkbox(value=False, description='Show |EMG|', indent=False, layout={'width': '130px'})
_pol_auto_toggle = ToggleButton(
    value=True, description='Auto y-scale', button_style='success',
    tooltip='Auto-scale shared y-axis from visible signals'
)
_pol_ymin = FloatText(value=_pol_ylim[0], description='Y min:', disabled=True, layout={'width': '145px'})
_pol_ymax = FloatText(value=_pol_ylim[1], description='Y max:', disabled=True, layout={'width': '145px'})

_pol_abs_cb.observe(_on_abs_change, names='value')
_pol_auto_toggle.observe(_on_auto_toggle, names='value')
_pol_ymin.observe(_on_ymin_change, names='value')
_pol_ymax.observe(_on_ymax_change, names='value')

_pol_controls = VBox([
    HBox([_pol_prev_btn, _pol_next_btn, _pol_page_drop, _pol_abs_cb]),
    HBox([_pol_auto_toggle, _pol_ymin, _pol_ymax]),
])

print(f"{len(_shared_amps)} shared amplitudes across {len(_pol_pages)} page(s), 2 per page.")
print("Top row: CORRECT polarity  |  Middle row: FLIPPED polarity  |  Bottom row: CONFIGAB polarity")
print("Note: FLIPPED traces should be the vertical mirror image of CORRECT traces.")
display(VBox([_pol_controls, _pol_out]))
_plot_pol_page(0)
'''